# Feature Engineering

In [45]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## Load Datasets

In [46]:
customers_df = pd.read_csv("../data/customers.csv")
orders_df = pd.read_csv("../data/orders.csv")

In [47]:
order_items_df = pd.read_csv("../data/order_items.csv")
payments_df = pd.read_csv("../data/payments.csv")

## Create Customer-Level Features

In [48]:
total_orders = (
    orders_df
    .groupby("customer_id")["order_id"]
    .count()
    .reset_index()
)

total_orders.rename(
    columns={"order_id": "total_orders"},
    inplace=True
)

total_orders.head()

,customer_id,total_orders
0,1,7
1,2,9
2,3,7
3,4,6
4,5,5


In [49]:
total_spent = (
    orders_df
    .groupby("customer_id")["total_amount"]
    .sum()
    .reset_index()
)

total_spent.rename(
    columns={"total_amount": "total_spent"},
    inplace=True
)

total_spent.head()

,customer_id,total_spent
0,1,515331.65
1,2,222108.65
2,3,605755.85
3,4,378409.60
4,5,87075.25


In [50]:
average_order_value = (
    orders_df
    .groupby("customer_id")["total_amount"]
    .mean()
    .round(2)
    .reset_index()
)

average_order_value.rename(
    columns={"total_amount": "average_order_value"},
    inplace=True
)

average_order_value.head()

,customer_id,average_order_value
0,1,73618.81
1,2,24678.74
2,3,86536.55
3,4,63068.27
4,5,17415.05


## Build Machine Learning Dataset

In [51]:
ml_df = customers_df.merge(
    total_orders,
    on="customer_id"
)

ml_df = ml_df.merge(
    total_spent,
    on="customer_id"
)

ml_df = ml_df.merge(
    average_order_value,
    on="customer_id"
)

ml_df.head()

,customer_id,customer_name,email,city,signup_date,gender,age,customer_segment,total_orders,total_spent,average_order_value
0,1,Yachana Khalsa,yachana.khalsa19@gmail.com,Chennai,2025-10-20,Male,21,Regular,7,515331.65,73618.81
1,2,Jeremiah Uppal,jeremiah.uppal21@gmail.com,Mumbai,2026-06-02,Male,44,Premium,9,222108.65,24678.74
2,3,Vaishnavi Sur,vaishnavi.sur71@gmail.com,Mumbai,2025-01-25,Female,21,Silver,7,605755.85,86536.55
3,4,Anamika Kothari,anamika.kothari58@gmail.com,Bhopal,2026-01-18,Female,34,Premium,6,378409.60,63068.27
4,5,Yashoda Khare,yashoda.khare46@gmail.com,Hyderabad,2026-03-13,Female,27,Silver,5,87075.25,17415.05


## Select Features and Target

In [52]:
ml_df.columns

Index(['customer_id', 'customer_name', 'email', 'city', 'signup_date',
       'gender', 'age', 'customer_segment', 'total_orders', 'total_spent',
       'average_order_value'],
      dtype='str')

In [53]:
ml_df = ml_df.drop(
    columns=[
        "customer_id",
        "customer_name",
        "email",
        "signup_date"
    ]
)

ml_df.head()

,city,gender,age,customer_segment,total_orders,total_spent,average_order_value
0,Chennai,Male,21,Regular,7,515331.65,73618.81
1,Mumbai,Male,44,Premium,9,222108.65,24678.74
2,Mumbai,Female,21,Silver,7,605755.85,86536.55
3,Bhopal,Female,34,Premium,6,378409.60,63068.27
4,Hyderabad,Female,27,Silver,5,87075.25,17415.05


## Encode Categorical Variables

In [54]:
ml_df = pd.get_dummies(
    ml_df,
    columns=[
        "gender",
        "city",
        "customer_segment"
    ],
    drop_first=True
)

ml_df.head()

,age,total_orders,total_spent,average_order_value,gender_Male,city_Bangalore,city_Bhopal,city_Chennai,city_Delhi,city_Hyderabad,city_Indore,city_Jaipur,city_Mumbai,city_Pune,customer_segment_Premium,customer_segment_Regular,customer_segment_Silver
0,21,7,515331.65,73618.81,True,False,False,True,False,False,False,False,False,False,False,True,False
1,44,9,222108.65,24678.74,True,False,False,False,False,False,False,False,True,False,True,False,False
2,21,7,605755.85,86536.55,False,False,False,False,False,False,False,False,True,False,False,False,True
3,34,6,378409.60,63068.27,False,False,True,False,False,False,False,False,False,False,True,False,False
4,27,5,87075.25,17415.05,False,False,False,False,False,True,False,False,False,False,False,False,True


In [55]:
bool_columns = ml_df.select_dtypes(include="bool").columns

ml_df[bool_columns] = ml_df[bool_columns].astype(int)

ml_df.head()

,age,total_orders,total_spent,average_order_value,gender_Male,city_Bangalore,city_Bhopal,city_Chennai,city_Delhi,city_Hyderabad,city_Indore,city_Jaipur,city_Mumbai,city_Pune,customer_segment_Premium,customer_segment_Regular,customer_segment_Silver
0,21,7,515331.65,73618.81,1,0,0,1,0,0,0,0,0,0,0,1,0
1,44,9,222108.65,24678.74,1,0,0,0,0,0,0,0,1,0,1,0,0
2,21,7,605755.85,86536.55,0,0,0,0,0,0,0,0,1,0,0,0,1
3,34,6,378409.60,63068.27,0,0,1,0,0,0,0,0,0,0,1,0,0
4,27,5,87075.25,17415.05,0,0,0,0,0,1,0,0,0,0,0,0,1


## Define Features and Target

In [56]:
X = ml_df.drop(
    columns=["total_spent"]
)

y = ml_df["total_spent"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (100, 16)
Target Shape: (100,)


## Save Processed Dataset

In [57]:
processed_df = pd.concat(
    [X, y],
    axis=1
)

processed_df.head()

,age,total_orders,average_order_value,gender_Male,city_Bangalore,city_Bhopal,city_Chennai,city_Delhi,city_Hyderabad,city_Indore,city_Jaipur,city_Mumbai,city_Pune,customer_segment_Premium,customer_segment_Regular,customer_segment_Silver,total_spent
0,21,7,73618.81,1,0,0,1,0,0,0,0,0,0,0,1,0,515331.65
1,44,9,24678.74,1,0,0,0,0,0,0,0,1,0,1,0,0,222108.65
2,21,7,86536.55,0,0,0,0,0,0,0,0,1,0,0,0,1,605755.85
3,34,6,63068.27,0,0,1,0,0,0,0,0,0,0,1,0,0,378409.60
4,27,5,17415.05,0,0,0,0,0,1,0,0,0,0,0,0,1,87075.25


In [58]:
processed_df.to_csv(
    "../data/ml_dataset.csv",
    index=False
)

print("ML dataset saved successfully!")

ML dataset saved successfully!
